# 07 — Refit Final Models on Full Data

Model selection (05a/05b/05c) trained on 2020–2024 and tested on 2025.
Those accuracy numbers are the honest reported metrics and stay unchanged.

Now that evaluation is done, we refit each winning model on the **full dataset**
(train + test combined) before saving the production pkl files.
This gives the models more examples to learn from — particularly helpful for
courses that only appear in 2025 test data.

**Output:** `models/final_model_offered.pkl`, `final_model_capacity.pkl`, `final_model_enrollment.pkl`

The `model_*.pkl` files from 05a/05b/05c are kept untouched as the evaluation artefacts.
The `final_model_*.pkl` files are what `src/predictor.py` will load.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import joblib
import json
from pathlib import Path
from sklearn.base import clone

DATA_PATH   = Path('../data/processed')
MODELS_PATH = Path('../models')

# load feature sets
with open(MODELS_PATH / 'feature_sets.json') as f:
    feature_sets = json.load(f)

# load train and test (unscaled — all winning models were tree-based)
train = pd.read_csv(DATA_PATH / '04_train.csv')
test  = pd.read_csv(DATA_PATH / '04_test.csv')
full  = pd.concat([train, test], ignore_index=True)

print(f'Train : {len(train):,} rows')
print(f'Test  : {len(test):,} rows')
print(f'Full  : {len(full):,} rows')

Train : 47,760 rows
Test  : 9,552 rows
Full  : 57,312 rows


## 2. Refit model_offered

In [2]:
FEATURES = feature_sets['model_offered']
TARGET   = 'was_offered'

# load the evaluated winner
winner = joblib.load(MODELS_PATH / 'model_offered.pkl')
print(f'Loaded: {winner.__class__.__name__}')

X_full = full[FEATURES]
y_full = full[TARGET]

# clone preserves hyperparameters, discards trained state
final_offered = clone(winner)
final_offered.fit(X_full, y_full)

joblib.dump(final_offered, MODELS_PATH / 'final_model_offered.pkl')
print(f'Saved : models/final_model_offered.pkl  (fit on {len(X_full):,} rows)')

Loaded: GradientBoostingClassifier
Saved : models/final_model_offered.pkl  (fit on 57,312 rows)


## 3. Refit model_capacity

In [3]:
FEATURES = feature_sets['model_capacity']
TARGET   = 'target_capacity'

winner = joblib.load(MODELS_PATH / 'model_capacity.pkl')
print(f'Loaded: {winner.__class__.__name__}')

# drop cold-start rows — no regression target to learn from
full_cap = full[full[TARGET].notna()]
X_full   = full_cap[FEATURES]
y_full   = full_cap[TARGET]

final_capacity = clone(winner)
final_capacity.fit(X_full, y_full)

joblib.dump(final_capacity, MODELS_PATH / 'final_model_capacity.pkl')
print(f'Saved : models/final_model_capacity.pkl  (fit on {len(X_full):,} rows, {len(full)-len(full_cap):,} dropped)')

Loaded: RandomForestRegressor
Saved : models/final_model_capacity.pkl  (fit on 24,083 rows, 33,229 dropped)


## 4. Refit model_enrollment

In [4]:
FEATURES = feature_sets['model_enrollment']
TARGET   = 'target_enrollment'

winner = joblib.load(MODELS_PATH / 'model_enrollment.pkl')
print(f'Loaded: {winner.__class__.__name__}')

full_enr = full[full[TARGET].notna()]
X_full   = full_enr[FEATURES]
y_full   = full_enr[TARGET]

final_enrollment = clone(winner)
final_enrollment.fit(X_full, y_full)

joblib.dump(final_enrollment, MODELS_PATH / 'final_model_enrollment.pkl')
print(f'Saved : models/final_model_enrollment.pkl  (fit on {len(X_full):,} rows, {len(full)-len(full_enr):,} dropped)')

Loaded: RandomForestRegressor
Saved : models/final_model_enrollment.pkl  (fit on 24,083 rows, 33,229 dropped)


## 5. Verify

In [5]:
files = [
    'final_model_offered.pkl',
    'final_model_capacity.pkl',
    'final_model_enrollment.pkl',
]

print('Final model files:')
for fname in files:
    path  = MODELS_PATH / fname
    model = joblib.load(path)
    size  = path.stat().st_size / 1024
    print(f'  {fname:40s}  {model.__class__.__name__:30s}  {size:.0f} KB')

print()
print('predictor.py should load final_model_*.pkl, not model_*.pkl')

Final model files:
  final_model_offered.pkl                   GradientBoostingClassifier      229 KB
  final_model_capacity.pkl                  RandomForestRegressor           143177 KB
  final_model_enrollment.pkl                RandomForestRegressor           179215 KB

predictor.py should load final_model_*.pkl, not model_*.pkl
